<a href="https://colab.research.google.com/github/JulianSantos-LATAMAI/ECON-5200/blob/main/Lab_6/%5BLab_6%5D_The_Architecture_of_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Data Ingestion (The Population)
df = sns.load_dataset('titanic')
print(f"Total Population: {len(df)}")
print(f"Population Survival Rate: {df['survived'].mean():.4f}")

# 2. Manual Shuffle (Simulation of Sampling)
np.random.seed(2026)
indices = np.random.permutation(len(df))  # Shuffle all row indices

# 3. Draw a Sample
sample_size = 100
sample_indices = indices[:sample_size]
sample = df.iloc[sample_indices]

print(f"\nSample Size: {len(sample)}")
print(f"Sample Survival Rate: {sample['survived'].mean():.4f}")

Total Population: 891
Population Survival Rate: 0.3838

Sample Size: 100
Sample Survival Rate: 0.4100


In [3]:
# 3. Cut the deck (80/20 Split)
split_point = int(len(df) * 0.80)  # ← was 80/20 = 4.0 (just a float, not useful)

# Slicing the shuffled indices
train_idx = indices[:split_point]
test_idx  = indices[split_point:]

# Creating the subsets
train_set = df.iloc[train_idx]
test_set  = df.iloc[test_idx]

# 4. Bias Check (The Delta)
train_surv = train_set['survived'].mean()
test_surv  = test_set['survived'].mean()
delta      = abs(train_surv - test_surv)

print(f"Train Survival Rate:   {train_surv:.4f}")
print(f"Test Survival Rate:    {test_surv:.4f}")
print(f"Sampling Bias (Delta): {delta:.4f}")

Train Survival Rate:   0.3736
Test Survival Rate:    0.4246
Sampling Bias (Delta): 0.0510


In [5]:
from sklearn.model_selection import train_test_split

# Stratify by 'pclass' ensures the distribution of classes is identical
X_train, X_test = train_test_split(
    df,
    test_size=0.20,
    random_state=2026,
    stratify=df['pclass']
)

print("\n--- Stratified Split ---")
print("Train Class Dist:\n", X_train['pclass'].value_counts(normalize=True))
print("Test Class Dist:\n",  X_test['pclass'].value_counts(normalize=True))


--- Stratified Split ---
Train Class Dist:
 pclass
3    0.550562
1    0.242978
2    0.206461
Name: proportion, dtype: float64
Test Class Dist:
 pclass
3    0.553073
1    0.240223
2    0.206704
Name: proportion, dtype: float64


In [6]:
from scipy.stats import chisquare

# 1. Define observed and expected arrays
observed = [450, 550]
expected = [500, 500]  # 50/50 split across 1000 users

# 2. Calculate Chi-Square statistic and p-value
chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"P-Value:              {p_value:.6f}")

# 3. Conclusion
if p_value < 0.01:
    print("\n🚨 CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.")
else:
    print("\n✅ Variance is within natural limits.")

Chi-Square Statistic: 10.0000
P-Value:              0.001565

🚨 CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.
